# Q3: Rain prediction

This notebook uses the five  examples to answer three questions with different algos:

1. What does Naive Bayes predict?
2. Which feature has the most information gain?
3. What weights does logistic regression learn?



In [ ]:
data = [
    ("Weak", "High", "Yes"),
    ("Strong", "High", "Yes"),
    ("Weak", "Low", "No"),
    ("Weak", "Low", "Yes"),
    ("Strong", "Low", "No"),
]

test_cases = [
    ("Weak", "High"),
    ("Strong", "High"),
    ("Weak", "Low"),
    ("Strong", "Low"),
]

for wind, humidity, rain in data:
    print(wind, humidity, rain)


Weak High Yes
Strong High Yes
Weak Low No
Weak Low Yes
Strong Low No


## 1. Naive Bayes

There are 3 Yes rows and 2 No rows, so the class priors are $P(Yes)=3/5$ and $P(No)=2/5$. We use **add-1 (Laplace) smoothing**, adding 1 to each category count, and adding 2 to the class total because each feature has two categories. This keeps probabilities from becoming zero.

For each test case:

$$\text{score}(c)=P(c)P(\text{Wind}\mid c)P(\text{Humidity}\mid c).$$

Divide each class score by the sum of both scores to get its posterior probability.


In [ ]:
alpha = 1

def class_count(label):
    count = 0
    for wind, humidity, rain in data:
        if rain == label:
            count += 1
    return count

def feature_probability(column, value, label):
    count = 0
    for row in data:
        if row[2] == label and row[column] == value:
            count += 1
    return (count + alpha) / (class_count(label) + 2 * alpha)

print("P(Yes) =", class_count("Yes") / len(data))
print("P(No)  =", class_count("No") / len(data))
for label in ("Yes", "No"):
    print("For", label, "rows:")
    print("  P(Weak)   =", feature_probability(0, "Weak", label))
    print("  P(Strong) =", feature_probability(0, "Strong", label))
    print("  P(High)   =", feature_probability(1, "High", label))
    print("  P(Low)    =", feature_probability(1, "Low", label))


P(Yes) = 0.6
P(No)  = 0.4
For Yes rows:
  P(Weak)   = 0.6
  P(Strong) = 0.4
  P(High)   = 0.6
  P(Low)    = 0.4
For No rows:
  P(Weak)   = 0.5
  P(Strong) = 0.5
  P(High)   = 0.25
  P(Low)    = 0.75


In [ ]:
nb_results = {}

print("Wind     Humidity   P(Yes)   P(No)    Prediction")
for wind, humidity in test_cases:
    yes_score = (class_count("Yes") / len(data)
                 * feature_probability(0, wind, "Yes")
                 * feature_probability(1, humidity, "Yes"))
    no_score = (class_count("No") / len(data)
                * feature_probability(0, wind, "No")
                * feature_probability(1, humidity, "No"))

    p_yes = yes_score / (yes_score + no_score)
    p_no = no_score / (yes_score + no_score)
    prediction = "Yes" if p_yes >= p_no else "No"
    nb_results[(wind, humidity)] = (p_yes, prediction)
    print(f"{wind:8} {humidity:10} {p_yes:.4f}   {p_no:.4f}   {prediction}")


Wind     Humidity   P(Yes)   P(No)    Prediction
Weak     High       0.8120   0.1880   Yes
Strong   High       0.7423   0.2577   Yes
Weak     Low        0.4898   0.5102   No
Strong   Low        0.3902   0.6098   No


## 2. Entropy and information gain

Entropy measures how mixed the Yes and No labels are. Information gain is the entropy before a split minus the average entropy after it. We calculate it once for Wind and once for Humidity.


In [ ]:
import math

def entropy(rows):
    total = len(rows)
    yes = sum(row[2] == "Yes" for row in rows)
    no = total - yes
    answer = 0
    for count in (yes, no):
        if count > 0:
            p = count / total
            answer -= p * math.log2(p)
    return answer

def information_gain(column):
    before = entropy(data)
    after = 0
    values = set(row[column] for row in data)
    for value in values:
        group = [row for row in data if row[column] == value]
        after += len(group) / len(data) * entropy(group)
    return before - after

wind_gain = information_gain(0)
humidity_gain = information_gain(1)
print(f"Entropy of Rain:             {entropy(data):.6f}")
print(f"Weighted entropy for Wind:   {entropy(data) - wind_gain:.6f}")
print(f"Weighted entropy for Humidity: {entropy(data) - humidity_gain:.6f}")
print(f"Information gain of Wind:     {wind_gain:.6f}")
print(f"Information gain of Humidity: {humidity_gain:.6f}")
print("Best first split:", "Humidity" if humidity_gain > wind_gain else "Wind")


Entropy of Rain:             0.970951
Weighted entropy for Wind:   0.950978
Weighted entropy for Humidity: 0.550978
Information gain of Wind:     0.019973
Information gain of Humidity: 0.419973
Best first split: Humidity


Humidity is the better first split. Both High-humidity examples say Yes. In the Low-humidity group, Strong/Low says No, while the two Weak/Low rows disagree.

## 3. Logistic regression

Encode Strong wind as $x_1=1$ and Weak wind as $x_1=0$. Encode High humidity as $x_2=1$ and Low humidity as $x_2=0$. The intercept input is $x_0=1$, and the target is $y=1$ for Yes and $y=0$ for No.

$$z=\theta_0+\theta_1x_1+\theta_2x_2,\qquad P(Yes)=\frac{1}{1+e^{-z}}.$$

We start all weights at zero. Each training step calculates predictions, adds up the errors, and adjusts the three weights. Our chosen settings are **L2 penalty $\lambda=0.25$**, learning rate **0.05**, and **20,000 steps**. The loss uses a sum over the five rows; only the two feature weights are penalized. There is no randomness.


In [ ]:
X = [
    [1, 0, 1],  # Weak, High
    [1, 1, 1],  # Strong, High
    [1, 0, 0],  # Weak, Low
    [1, 0, 0],  # Weak, Low
    [1, 1, 0],  # Strong, Low
]
y = [1, 1, 0, 1, 0]

regularization = 0.25
learning_rate = 0.05
steps = 20000
weights = [0.0, 0.0, 0.0]

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

for step in range(steps):
    gradient = [0.0, 0.0, 0.0]

    for x, correct_answer in zip(X, y):
        z = weights[0] * x[0] + weights[1] * x[1] + weights[2] * x[2]
        error = sigmoid(z) - correct_answer
        for j in range(3):
            gradient[j] += error * x[j]

    gradient[1] += regularization * weights[1]
    gradient[2] += regularization * weights[2]
    for j in range(3):
        weights[j] -= learning_rate * gradient[j]

print(f"Intercept:       {weights[0]:+.6f}")
print(f"Wind weight:     {weights[1]:+.6f}")
print(f"Humidity weight: {weights[2]:+.6f}")


Intercept:       +0.048624
Wind weight:     -0.539200
Humidity weight: +1.616261


### Logistic-regression predictions

We predict Yes if $P(Yes)\geq0.5$. The drone does not deliver when rain is predicted; otherwise, it delivers.


In [ ]:
lr_results = {}

print("Wind     Humidity   z         P(Yes)   Prediction   Drone action")
for wind, humidity in test_cases:
    x1 = 1 if wind == "Strong" else 0
    x2 = 1 if humidity == "High" else 0
    z = weights[0] + weights[1] * x1 + weights[2] * x2
    p_yes = sigmoid(z)
    prediction = "Yes" if p_yes >= 0.5 else "No"
    lr_results[(wind, humidity)] = (p_yes, prediction)
    action = "Do not deliver" if prediction == "Yes" else "Deliver"
    print(f"{wind:8} {humidity:10} {z:+.6f} {p_yes:.4f}   {prediction:10}   {action}")


Wind     Humidity   z         P(Yes)   Prediction   Drone action
Weak     High       +1.664886 0.8409   Yes          Do not deliver
Strong   High       +1.125686 0.7550   Yes          Do not deliver
Weak     Low        +0.048624 0.5122   Yes          Do not deliver
Strong   Low        -0.490575 0.3798   No           Deliver


## 4. Compare the methods

Naive Bayes and logistic regression use the same data but different probability formulas. Information gain identifies the stronger feature; it does not directly predict a probability.
